# Domain Decomposition 

In this tutorial we present how to split a global domain into multiple subdomains, for the implementation of domain decomposition with [PyGeoN](https://github.com/compgeo-mox/pygeon). The problem to solve will be Darcy flow in a mixed formulation, with the unknowns pressure $p$ and flux $q$.

Let $\Omega=(0,1)^3$ with outer boundary $\partial \Omega$, be split into 4 non-overlapping subdomains ${\Omega}_l$ with interfaces ${\Gamma}_l$ and outward unit normal ${\nu}_l$. Given 
$k_l$ the matrix permeability and $g_l$ a vector source term, we want to solve the following problem on each subdomain: find $({q}_l, p_l)$ such that
$$
\left\{
\begin{array}{ll}
\begin{array}{l} 
k^{-1}_l {q}_l + \nabla p_l = {g}_l\\
\nabla \cdot {q}_l = 0
\end{array}
&\text{in } \Omega
\end{array}
\right.
$$
with boundary conditions:
$$ p = 0 \text{ on } \partial_{top} \Omega \qquad p = 1 \text{ on } \partial_{bottom} \Omega \qquad \nu \cdot q = 0 \text{ on } \partial_{left} \Omega \cup \partial_{right} \Omega$$




We present *step-by-step* how to create the grid, declare the problem data, and finally solve the problem.

First we import some of the standard modules, like `numpy` and `scipy.sparse`. Since PyGeoN is based on [PorePy](https://github.com/pmgbergen/porepy) we import both modules.

In [ ]:
import os
import shutil
import numpy as np
import scipy.sparse as sps
from scipy.sparse.csgraph import connected_components

import porepy as pp
import pygeon as pg

import inspect

We create now the grid, since we will use a Lagrangian of order 1 to approximation for ${u}$ we are restricted to simplices. In this example we consider a 2-dimensional structured grid, but the presented code will work also in 1d and 3d. PyGeoN works with mixed-dimensional grids, so we need to convert the grid.

In [ ]:
mesh_size = 0.3     # mesh size
dim = 3             # dimension of the domain  
key = "pressure"    # key for the unknown variable

# Define a rectangular domain in terms of range in the three dimensions
bbox = {"xmin": 0, "xmax": 1, "ymin": 0, "ymax": 1, "zmin": 0, "zmax": 1}
domain = pp.Domain(bounding_box=bbox)

# Next, define the placement of the interfaces (fractures in PorePy). The 
# fractures are defined by their corner points.
interface_1 = pp.PlaneFracture(
    np.array([[0, 1, 1, 0], [0, 0, 1, 1], [0.5, 0.5, 0.5, 0.5]])
)
interface_2 = pp.PlaneFracture(
    np.array([[0.5, 0.5, 0.5, 0.5], [0, 1, 1, 0], [0, 0, 1, 1]])
)
interfaces = [interface_1, interface_2]

# The mixed dimensional grid is then created
mdg = pg.grid_from_domain(
    domain,
    mesh_size=mesh_size,
    fractures=interfaces,
    dim=dim,
)
mdg.compute_geometry()

# For 2D domains, the grid can be visualized using PorePy's "pp.plot_grid(mdg)",
# but for 3D domains, visualization should be done in ex. ParaView by 
# exporting: pp.Exporter(mdg, "mixed_dimensional_grid").write_vtu()

# Set up global domain:
sd, data = mdg.subdomains(dim=3, return_data=True)[0]

In [ ]:
# initialize the parameters on the grid
perm = pp.SecondOrderTensor(np.full(sd.num_cells, K))

param = {"second_order_tensor": perm}
pp.initialize_data(sd, data, key, param)

In [ ]:
# with the following steps we identify the portions of the boundary
# to impose the boundary conditions
x, y, z = sd.face_centers[0, :], sd.face_centers[1, :], sd.face_centers[2, :]

top_mask = z == 1.0
side_1_mask = y == 0.0
side_2_mask = x == 0.0
side_3_mask = y == 1.0
side_4_mask = x == 1.0
bottom_mask = z == 0.0

# set flags for essential and natural boundary conditions
nat_bc_flags = np.logical_or(
    np.logical_or(
        np.logical_or(side_1_mask, side_2_mask),
        np.logical_or(side_3_mask, side_4_mask),
    ),
    np.logical_or(top_mask, bottom_mask),
)

ess_bc_flags = np.logical_or(
    np.logical_or(
        np.logical_or(side_1_mask, side_2_mask),
        np.logical_or(side_3_mask, side_4_mask),
    ),
    np.logical_or(top_mask, bottom_mask),
)

In [3]:
def first_order_tensor(grid,
        vx: np.ndarray,
        vy: np.ndarray = None,
        vz: np.ndarray = None,
    ):
    
        n_cells = vx.size
        vel = np.zeros((3, n_cells))

        vel[0, ::] = vx
             
        if vy is not None:
            vel[1, ::] = vy

        if vz is not None:
            vel[2, ::] = vz

        return vel  

In [ ]:

start_time = time.perf_counter()
dt = end_time / num_steps

mdg, nat_bc_flags, ess_bc_flags = solver.create_grid(
    mesh_size, dim, K
)

sd, data = mdg.subdomains(dim=3, return_data="True")[0]
n_cells = sd.num_cells
n_faces = sd.num_faces

n_interface_cells = sum(
    interface.num_cells for interface in mdg.subdomains(dim=2)
)

face_pairs = mdg.subdomains(dim=3)[0].frac_pairs.T

M_blocks = []

for interface in mdg.subdomains(dim=2):
    M_loc = P0.assemble_mass_matrix(interface)
    M_blocks.append(M_loc)

M_int = sps.block_diag(M_blocks, format="csc")

# Construct the constant global matrices
M_p = P0.assemble_mass_matrix(sd)
M_q = RT0.assemble_mass_matrix(sd)

div = M_p @ RT0.assemble_diff_matrix(sd)
jump = -pg.div(mdg)[n_cells : n_cells + n_interface_cells, :n_faces]

C = M_int @ jump

B_full = jump.T @ M_int @ jump
B = sps.csc_array(sps.diags(B_full.diagonal()))
B.eliminate_zeros()

grav = M_q @ RT0.interpolate(sd, grav_func)

# get the degrees of freedom for u
dof_p, dof_q = div.shape

_, labels = connected_components(
    M_q, directed=False, connection="weak"
)

ess_bc_vals = RT0.interpolate(sd, ess_bound_cond)

# assemble the time-independent right-hand side
rhs_const = np.zeros(dof_q + dof_p)
rhs_const[: dof_q] += grav

proj_p = P0.eval_at_cell_centers(sd)
proj_q = RT0.eval_at_cell_centers(sd)

# set and store initial conditions
p = P0.interpolate(sd, init_p_cond)
q = RT0.interpolate(sd, init_q_cond)
g = g_init(q, p)

print(f"Grid size: {mesh_size}")

# set natural boundary values
nat_bc_vals = RT0.assemble_nat_bc(
    sd,
    lambda X: solution_p(X[0], X[1], X[2], n * dt),
    nat_bc_flags,
)

# assemble the time-dependent right-hand side
rhs_fixed = rhs_const.copy()
rhs_fixed[: dof_q] -= nat_bc_vals

p_prev = p.copy()
q_prev = q.copy()
g_prev = g.copy()

for i in range(dom_iter):
    x = np.zeros(dof_q + dof_p)

    for dom in range(max(labels) + 1):
        C_dom = C.copy()
        C_dom = C_dom[:, labels == dom]
        int_vals = C_dom.T @ g[:, dom]
        rhs_dom = rhs_fixed.copy()
        rhs_dom[: dof_q][labels == dom] -= int_vals

        
        p_cell = proj_p @ p_prev
        q_cell = proj_q @ q_prev

        M_q = RT0.assemble_mass_matrix(sd, data)

        # iterative rhs
        rhs_global = rhs_fixed.copy()
        rhs_global[dof_q :] += L * M_p @ p_prev - theta

        # saddle-point system
        spp_global = sps.block_array(
            [
                [M_q + c * B, -div.T],
                [dt * div, L * M_p],
            ],
            format="csc",
        )
        _, labels = connected_components(spp_global, directed=False, connection="weak")

        # Global index of the degrees of freedom for q
        dof_id = np.where(labels == dom_num)[0]
        spp = spp_global[dof_id, :][:, dof_id]
        rhs = rhs_global[dof_id].copy()

        # flag the essential boundary conditions
        full_ess_bc_flags = np.concatenate(
            [ess_bc_flags, np.zeros(dof_p, dtype=bool)]
        )[dof_id]

        full_ess_bc_vals = np.concatenate([ess_bc_vals, np.zeros(dof_p)])[dof_id]

        ls = pg.LinearSystem(spp, rhs)

        ls.flag_ess_bc(full_ess_bc_flags, full_ess_bc_vals)

        # solve the problem
        x = ls.solve()

        x_full = np.zeros(dof_q + dof_p)
        x_full[dof_id] = x

        # extract the variables
        p = x_full[dof_q :]
        q = x_full[: dof_q]

        jump_error = np.sqrt(q.T @ B_full @ q)

        # Log message with error and current iteration
        print(
            "Domain decomposition iteration #"
            + str(i + 1)
            + ", jump error: "
            + str(jump_error)
        )

        if jump_error <= int_tol:
            break
        else:
            g = g(g_prev, q)
            g_prev = g.copy()
            p_prev = p.copy()
            q_prev = q.copy()

With the following code we set the data, in particular the velocity field, diffusion coefficient and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need few steps.

We define the source term function. Given below are two source functions: a point source, and a gaussian source at the center cell of the grid.

In [4]:
def source_term_gauss(x):
    # Example: a Gaussian source at the center of the grid
    center = sd.cell_centers[:,sd.num_cells // 2]
    sigma = 0.05
    r2 = np.sum((x - center)**2)
    return np.exp(-r2 / (2 * sigma**2))

def source_term_point(x):
    # Example: a point source at the center of the grid

    # Define the center cell
    center_cell = sd.num_cells // 2 - 1 
    # Get the boundary nodes of the grid
    bd_nodes = sd.get_all_boundary_nodes()
    # A map from cell to nodes
    node_map = sd.cell_nodes()
    # Get the nodes of the center cell
    center_nodes = node_map[:,center_cell].nonzero()[0]
    # Filter out the boundary nodes from the center nodes
    mask = ~np.isin(center_nodes, bd_nodes)
    center_nodes = center_nodes[mask]
    # Get the coordinates of the center nodes
    source_nodes_coord = sd.nodes.T[center_nodes]

    # Check if the input x matches any of the center nodes' coordinates
    return 1.0 if np.any(np.all(source_nodes_coord == x, axis=1)) else 0.0

Define the natural boundary condition function

In [5]:
def u_bc(x):
    # Example: Neumann boundary condition with inflow at the left and zero 
    # outflow elsewhere
    return 1.0 if abs(x[0]) < 1e-10 else 0.0

In [6]:
key = "mass"

nat_bc_val = []
ess_bc = []
source = []
sol = []

# Use first order Lagrange elements to approximate the solution
P1 = pg.Lagrange1(key)

# Loop through each subdomain in the mixed-dimensional grid
# (NB: only one subdomain in this case)
for sd, data in mdg.subdomains(return_data=True):
    # Set the values for the diffusion tensor and velocity field parameter 
    diff = pp.SecondOrderTensor(np.ones(sd.num_cells))
    vel_field = first_order_tensor(sd, np.ones(sd.num_cells))
    # Store the parameters in the subdomain data
    param = {"first_order_tensor": vel_field, "second_order_tensor": diff}
    pp.initialize_data(sd, data, key, param)

    # Identify boundary faces
    left = sd.face_centers[0, :] == 0
    right = sd.face_centers[0, :] == 1
    bottom = sd.nodes[1, :] == 0
    top = sd.nodes[1, :] == 1
    
    # Define natural and essential boundary faces 
    nat_bc_faces = np.logical_or(left, right)
    ess_bc_faces = np.logical_or(bottom, top)
    
    # Set values for the natural boundary condition
    nat_bc_val.append(dt * P1.assemble_nat_bc(sd, u_bc, nat_bc_faces))

    # Set boolean array for nodes which are essential boundary condition
    ess_bc.append(ess_bc_faces)

    # Set the source term values 
    mass = P1.assemble_mass_matrix(sd)
    source.append(dt * mass @ P1.interpolate(sd, source_term_point))

Once the data are assigned to the mixed-dimensional grid, we construct the matrices. In particular, the linear system given in above. Once the matrix is created, we also construct the right-hand side containing the boundary conditions.

In [7]:
# Construct the local matrices
mass = P1.assemble_mass_matrix(sd)
adv = P1.assemble_adv_matrix(sd, data)
stiff = P1.assemble_stiff_matrix(sd, data)

# Assemble the global matrix
# fmt: off
global_matrix = mass + dt*(adv + stiff)
# fmt: on

# Get the degrees of freedom for u
dof_u = P1.ndof(sd)

# Assemble the time-independent right-hand side
rhs_const = np.zeros(dof_u)
rhs_const[:dof_u] += np.hstack(nat_bc_val) + np.hstack(source)

# Set initial conditions
u = np.zeros(dof_u)

# Add the initial condition to the time series
sol.append(u)

KeyError: 'vector_field'

We need to solve the linear system, PyGeoN provides a framework for that. The actual imposition of essential boundary conditions (flux boundary conditions) might change the symmetry of the global system, the class `pg.LinearSystem` preserves this structure by internally eliminating these degrees of freedom. Once the problem is solved, we extract the solutions $u$

In [ ]:
for n in np.arange(num_steps):

    # Update the right-hand side with the current solution
    rhs = rhs_const.copy()
    rhs[:dof_u] += np.hstack(mass @ u)

    # Set up the linear solver
    ls = pg.LinearSystem(global_matrix, rhs)

    # Flag the essential boundary conditions
    ls.flag_ess_bc(np.hstack(ess_bc), np.zeros(dof_u))

    # Solve the linear system
    x = ls.solve()

    # Extract the variables
    u = x[:dof_u]
    
    # Add the solution to the time series
    sol.append(u)

Since the computed $u$ is one value per node of the grid, for visualization purposes we project the mass in each cell center. We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
output_directory = os.path.join(os.path.dirname(__file__), "adv-diff sol")
# Delete the output directory, if it exisis
if os.path.exists(output_directory):
    shutil.rmtree(output_directory)

save = pp.Exporter(mdg, "adv-diff", folder_name=output_directory)

proj_u = P1.eval_at_cell_centers(sd)

for n, u in enumerate(sol):
    for _, data in mdg.subdomains(return_data=True):
        
        # post process variables
        cell_u = proj_u @ u

        pp.set_solution_values("mass", cell_u, data, time_step_index=0)
        save.write_vtu(["mass"], time_step=n)

save.write_pvd(range(len(sol)))